# Pangenome Analysis Notebook

This notebook walks through a GPU accelerated pangenome analysis workflow using NVIDIA Parabricks. We start by downloading a pangenome graph and using filters to extract the reference path. Next we use the Parabricks Giraffe aligner to align FASTQ samples to this reference. Finally, we run the aligned samples through Parabricks DeepVariant for variant calling. 

### What is a pangenome? 

A pangenome is a comprehensive collection of all genomic sequences and variations present across multiple individuals or populations of a species, rather than relying on a single reference genome. Instead of treating one genome as the standard reference, pangenomes capture the genetic diversity by including sequences from many individuals, making it easier to identify variants and structural differences that might be missed in traditional single-reference approaches. Using pangenomes leads to better alignment and more accurate variant calling. 


## Step 1: Download the data

For this workflow, we start by downloading the FASTQ samples and the pangenome reference. 

Note: This will take about 30 mins to download. 

In [ ]:
import os

# Set the path to the data directory 
os.environ['DATA_DIR'] = "/ephemeral"
print("✅ HOST_PATH exported to shell")

In [ ]:
%%sh 

../scripts/download_pangenome.sh $DATA_DIR

### Pangenome graph 

Pangenome data can stored in a few different ways, as outlined in the figure below. For our purposes we will use a pangenome graph. 

Source: [A gentle introduction to pangenomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC11570541/#f2) 

![fq2bam_diagram](../images/pangenome_storage.png)

Specifically, we use the [HPRC](https://humanpangenome.org/) (Human Pangenome Reference Consortium) version 1.1 graph, which was constructed using [Minigraph-Cactus](https://github.com/ComparativeGenomicsToolkit/cactus/blob/master/doc/pangenome.md). We need the following files: 
- `.gbz` - The compressed graph 
- `.dist` - The distance index used for computing distances between points in the graph 
- `.min` - The minimizer index used to find minimizer substrings in the graph 

Now we must identify the reference path, which is the path through the graph that matches the human standard reference GRCh38 ([Genome Reference Consortium Human Build 38](https://genome.ucsc.edu/cgi-bin/hgGateway?token=0.eDjpSlexMn1Xh8rGqvYUpctc_8ZQwgdWcTDP0mg0e5YHk6b1gPw7TiSXXxKFNDG0F4gn8voyGzzIBzNT6aHsO2cylGlCg2UgoFVsH2vv9Qt2PBk7j3yHE-Bqey5qPNprFVirMzNU7meyTNF2r4qu6kwcgDgZEXC6lw_kgMuQfqkYxmhkgVOGgKfq7V3e7ixhPigalMgvKBCznLEf-qm3pWI74qDj-hOWHGBCykTmP9vd4Mj9fK8AAzLkY6LOVyr9lewYtUW9A-CRaa_BIUpoPxTOcj7QEkaLdmCn1pJytLmrkQXW64XSadhQzyI7REDt4M2-Lj3f6GkMMswIEG1ZOHJfBaK8UFPV1VaKzYduFCMr0z42RH4eg9qGQjdSPJxfMfK3bsydrMLcEba114kZWUJUG4GErA5ZW-y5kvjbwuh0osLhY9laHhdsPOBa61g8gRE_el82HxLFcX0-AuKOjjP7MGZJtJr08bu8_LxyWROLtMYOKW8IAA9sb79_0rHChM-TUxs96jAWXKjuVAzm_USOT1MFNSeDGeBmJ4kIrJkyoJB3wdQmLA4UaH83QW2q_VFXlgtwJjwY-ZY48HycsjVpZD_hhUWk_pJDuECJyazd5rOLVoWRCQzy4202zZsV5gUO9kaJqqsKe6dQfRz1RyIXoj_cxwcRs2ZZftF5N_KIXRQ_UduCJdAavB1kd_lh_QiBYieL-uUzLgMRwf_n1HCqZPgvduixF_N__-NOUEuz1j_7hZU3aTCv5x-i7oQMtluO2mVkSE8Vj8zIkpxB0404yKI4hLzKNfDj3KNjvCdR-XbksnaYjiuncgNwXiSroiScC_fwMubmv7Vj3Rm9DIEU7Ld3tXVkgh_SYYs2aTB34z3teD9ewBFJScb2Rgj_RfEnSWbC-IZjVZui5_B6AefNkQShaR4tIQM76iYA8Z8.NV4fHkVJjzg_jyWUdf5ORQ.5c8945b5c8156b8b333c2f2e4ac1371136e953526116bb6062a3d89aadefef88)). This helps to map our graph into a standard coordinate system that is used by other downstream tools and maintains compatability with tools that don't fully support graphs. 

For this we use the [`vg`](https://github.com/vgteam/vg) software package which has a tool called `paths`. The code cell below will find all the paths that start with `GRCh38` and save them to a file. 

In [ ]:
%%sh 

docker run --rm \
    --volume $DATA_DIR:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x ref/hprc-v1.1-mc-grch38.gbz \
        -L \
        -Q GRCh38 > $DATA_DIR/ref/hprc-v1.1-mc-grch38.paths

Next we use `grep` to further filter this list to remove any unwanted paths. In this case, we filter out anything that's not Chromosomes 1-22 or Chromosomes X and Y. 

In [ ]:
%%sh 

grep -v _decoy $DATA_DIR/ref/hprc-v1.1-mc-grch38.paths \
    | grep -v _random \
    | grep -v chrUn_ \
    | grep -v chrEBV \
    | grep -v chrM \
    | grep -v chain_ > $DATA_DIR/ref/hprc-v1.1-mc-grch38.paths.sub

Now we have the data downloaded and pre-processed, the data directory should have these files:  

```
data
├── ref
│   ├── hprc-v1.1-mc-grch38.dist
│   ├── hprc-v1.1-mc-grch38.gbz
│   ├── hprc-v1.1-mc-grch38.min
│   ├── hprc-v1.1-mc-grch38.paths
│   └── hprc-v1.1-mc-grch38.paths.sub
├── test_1.fastq.gz
└── test_2.fastq.gz

2 directories, 7 files
```

## Giraffe

Now that the samples are downloaded and the graph has been pre-processed, we are ready for short-read alignment using Giraffe. 

Giraffe was originally developed by Dr. Benedict Paten's lab at the University of California, Santa Cruz (UCSC). It brings many improvements over previous mapping methods, including haplotype-aware mapping, pre-calculate indexes, and now hardware acceleration on the GPU. Combined these allow for faster and more accurate results. For more details on Giraffe, see the [original paper](https://www.science.org/doi/10.1126/science.abg8871). 

In the next code cell, we will run the GPU-accelerated [Parabricks Giraffe](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_giraffe.html). 

In [ ]:
%%sh 

# Make the output directory if it doesn't exist 
mkdir -p $DATA_DIR/out

# Run GPU-accelerated Giraffe using Parabricks 
docker run --rm --gpus all \
    --volume $DATA_DIR:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun giraffe \
        --read-group "sample_rg1" \
        --sample "sample-name" \
        --read-group-library "library" \
        --read-group-platform "platform" \
        --read-group-pu "pu" \
        --dist-name ref/hprc-v1.1-mc-grch38.dist \
        --minimizer-name ref/hprc-v1.1-mc-grch38.min \
        --gbz-name ref/hprc-v1.1-mc-grch38.gbz \
        --ref-paths ref/hprc-v1.1-mc-grch38.paths.sub \
        --in-fq sample_1.fq.gz sample_2.fq.gz \
        --out-bam out/sample.bam

There are new files in the output directory:

- `sample.bam` - the aligned samples 
- `sample.bam.bai` - an index into the aligned samples
- `sample_chrs.txt` - each contig/chromosome and how many reads were mapped to it

In [ ]:
%%sh 

ls -lh $DATA_DIR/out

## Pangenome-Aware DeepVariant

In this last section of the notebook, we run variant calling on our aligned samples. 

Since our aligned was run using a pangenome graph, but DeepVariant requires a FASTA reference file, we must create a FASTA from our graph. For this we can again use `vg paths`: 

In [ ]:
%%sh 

# Extract the sequences corrresponding to the list of paths to a FASTA file
docker run --rm \
    --volume $DATA_DIR/ref:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths \
        -x hprc-v1.1-mc-grch38.gbz \
        -p hprc-v1.1-mc-grch38.paths.sub \
        -F > $DATA_DIR/ref/hprc-v1.1-mc-grch38.fa

This FASTA needs to be indexed, so we use `samtools faidx` as we would for a normal linear reference. 

In [ ]:
%%sh 

samtools faidx $DATA_DIR/ref/hprc-v1.1-mc-grch38.fa

Now we are ready to run the GPU-accelerated [Parabricks Pangenome-Aware DeepVariant](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_pangenome_aware_deepvariant.html). 

In [ ]:
%%sh 

# Run GPU-accelerated Pangenome-Aware DeepVariant using Parabricks
docker run --rm --gpus all \
    --volume $DATA_DIR:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun pangenome_aware_deepvariant \
    --ref ref/hprc-v1.1-mc-grch38.fa \
    --pangenome ref/hprc-v1.1-mc-grch38.gbz \
    --in-bam out/sample.bam \
    --out-variants out/sample.vcf

We should now see `sample.vcf` in the output directory. This file contains all the information about which positions in the genome express variations from the reference genome and how those variants were filtered. 

In [ ]:
%%sh 

ls -lh $DATA_DIR/out

We can even inspect the `vcf` using [`bcftools`](https://samtools.github.io/bcftools/bcftools.html) to make sure it was properly written: 

In [ ]:
%%sh 

bcftools view --no-header $DATA_DIR/out/sample.vcf | head

## Conclusion 

This notebook demonstrated a complete GPU-accelerated pangenome analysis workflow using NVIDIA Parabricks. We downloaded the HPRC pangenome graph, aligned short-read sequencing samples using the Giraffe aligner, and performed variant calling with Pangenome-Aware DeepVariant. By leveraging pangenome graphs instead of linear reference genomes, this approach captures genetic diversity across populations, leading to improved alignment accuracy and more comprehensive variant detection. The GPU acceleration provided by Parabricks significantly reduces computational time for these computationally intensive genomics tasks.

For more information on GPU accelerated genomics workflows, check out the Parabricks [documentation](https://docs.nvidia.com/clara/parabricks/latest/index.html). 